# Chapter 30 : The Prompt Evaluation : A/B Testing Prompts


Now we'll learn how to measure which prompt performs better instead of choosing one based on intuition.

#  What is prompt evaluation?
Suppose you have two prompts for sentiment classification.
Prompt A — Simple
Classify the sentiment as Positive,
Negative, or Neutral.




Prompt B — Detailed
Classify the sentiment as Positive,
Negative, or Neutral.

Positive = satisfaction or praise.
Negative = dissatisfaction or criticism.
Neutral = factual text without clear sentiment.

Return only one label.




Which is better? We can't know just by reading the prompts. We need to test both on the same examples and compare their predictions against known answers.
2. Our evaluation workflow
Create a labeled test dataset
Run Prompt A and Prompt B on the same texts
Compare predictions with expected labels
Calculate accuracy and choose based on evidence

In [1]:
from dotenv import load_dotenv
load_dotenv()
import os

groq_api_key = os.getenv("GROQ_API_KEY")

In [2]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="openai/gpt-oss-20b",
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ["GROQ_API_KEY"])

/Users/vimal/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [5]:
test_data = [
    {
        "text": "I absolutely love this application.",
        "expected": "POSITIVE"
    },
    {
        "text": "The service was terrible.",
        "expected": "NEGATIVE"
    },
    {
        "text": "The package arrived on Tuesday.",
        "expected": "NEUTRAL"
    },
    {
        "text": "Customer support was helpful and friendly.",
        "expected": "POSITIVE"
    },
    {
        "text": "I am disappointed with the product.",
        "expected": "NEGATIVE"
    },
    {
        "text": "The meeting starts at 10 AM.",
        "expected": "NEUTRAL"
    },
    {
        "text": "Excellent quality and fast delivery.",
        "expected": "POSITIVE"
    },
    {
        "text": "The application keeps crashing.",
        "expected": "NEGATIVE"
    }
]

In [6]:
# Defining two prompts 
prompt_a = """
Classify the sentiment of this text as
POSITIVE, NEGATIVE, or NEUTRAL.

Return only the label.

Text: {text}
"""

prompt_b = """
You are a sentiment classification system.

Classify the text using these definitions:
- POSITIVE: satisfaction, praise, or favorable opinion.
- NEGATIVE: dissatisfaction, criticism, or unfavorable opinion.
- NEUTRAL: factual statement without clear sentiment.

Return exactly one label:
POSITIVE, NEGATIVE, or NEUTRAL.

Do not explain your answer.

Text: {text}"""

In [23]:
import time
def evaluate_prompt(template, test_data):
    results = []

    for item in test_data:
        prompt = template.format(text = item["text"])

        try: 
            start_time = time.time()
            response = llm.invoke(prompt)
            latency = round(time.time() - start_time, 3)
            prediction = response.content.strip()

            valid_labels = [ "POSITIVE","NEGATIVE","NEUTRAL"]

            if prediction not in valid_labels:
                prediction = "INVALID"

            results.append({
                "text" : item["text"],
                "expected" : prediction,
                "predicted" : prediction == item["expected"],
                "latency_seconds" : latency,
                "correct": prediction == item["expected"],
                "total_tokens" : (response.usage_metadata)
            })
        except Exception as e:
            results.append({
                "text" : item["text"],
                "expected" : item["expected"],
                "predicted" : "Error",
                "correct" : False,
                "error" : str(e)
            })
    correct_count = sum(row["correct"] for row in results)

    accuracy = correct_count/len(test_data) * 100

    return { 
        "accuracy_percent" : round(accuracy,2),
        "correct" : correct_count,
        "total" : len(test_data),
        "result" : results
    }

result_a = evaluate_prompt(prompt_a, test_data)
result_b = evaluate_prompt(prompt_b, test_data)


        


In [24]:
print(result_a)

{'accuracy_percent': 100.0, 'correct': 8, 'total': 8, 'result': [{'text': 'I absolutely love this application.', 'expected': 'POSITIVE', 'predicted': True, 'latency_seconds': 0.571, 'correct': True, 'total_tokens': {'input_tokens': 105, 'output_tokens': 41, 'total_tokens': 146, 'input_token_details': {}, 'output_token_details': {'reasoning': 30}}}, {'text': 'The service was terrible.', 'expected': 'NEGATIVE', 'predicted': True, 'latency_seconds': 0.423, 'correct': True, 'total_tokens': {'input_tokens': 104, 'output_tokens': 30, 'total_tokens': 134, 'input_token_details': {}, 'output_token_details': {'reasoning': 19}}}, {'text': 'The package arrived on Tuesday.', 'expected': 'NEUTRAL', 'predicted': True, 'latency_seconds': 0.455, 'correct': True, 'total_tokens': {'input_tokens': 105, 'output_tokens': 45, 'total_tokens': 150, 'input_token_details': {}, 'output_token_details': {'reasoning': 33}}}, {'text': 'Customer support was helpful and friendly.', 'expected': 'POSITIVE', 'predicted': 

In [25]:
print(result_a)

{'accuracy_percent': 100.0, 'correct': 8, 'total': 8, 'result': [{'text': 'I absolutely love this application.', 'expected': 'POSITIVE', 'predicted': True, 'latency_seconds': 0.571, 'correct': True, 'total_tokens': {'input_tokens': 105, 'output_tokens': 41, 'total_tokens': 146, 'input_token_details': {}, 'output_token_details': {'reasoning': 30}}}, {'text': 'The service was terrible.', 'expected': 'NEGATIVE', 'predicted': True, 'latency_seconds': 0.423, 'correct': True, 'total_tokens': {'input_tokens': 104, 'output_tokens': 30, 'total_tokens': 134, 'input_token_details': {}, 'output_token_details': {'reasoning': 19}}}, {'text': 'The package arrived on Tuesday.', 'expected': 'NEUTRAL', 'predicted': True, 'latency_seconds': 0.455, 'correct': True, 'total_tokens': {'input_tokens': 105, 'output_tokens': 45, 'total_tokens': 150, 'input_token_details': {}, 'output_token_details': {'reasoning': 33}}}, {'text': 'Customer support was helpful and friendly.', 'expected': 'POSITIVE', 'predicted': 

In [26]:
print("PROMPT A ACCURACY:", result_a["accuracy_percent"], "%")
print("PROMPT B ACCURACY:", result_b["accuracy_percent"], "%")

PROMPT A ACCURACY: 100.0 %
PROMPT B ACCURACY: 100.0 %
